# SQL vs Excel: проверка количества ИНН

Тетрадка считает в SQL:
- `sql_agr_cnt` (уникальные договоры),
- `sql_inn_cnt` (уникальные ИНН)

в периметре DAG за выбранный месяц.

In [ ]:
month_start = "2026-02-01"  # YYYY-MM-01

In [ ]:
with imp:
    sql_cnt = imp.fetch(f"""
        with params as (
            select cast('{month_start}' as date) as month_start
        ),
        feb_clients as (
            select distinct
                c.c_inn as inn,
                a.abs_agr_id as agr_id
            from ods_alpha.scd1_agreements a
            join ods_alpha.scd1_companies c
              on c.n_cmp = a.n_cmp_client
            join ods_alpha.scd1_trx_acq ta
              on ta.n_agr = a.n_agr
            join ods_alpha.scd1_trx t
              on t.n_trx = ta.n_trx
            join params p on 1=1
            where a.acq_class = 'SA'
              and a.abs_agr_id is not null
              and t.c_trx_class = 'SA'
              and t.c_trx_type = 'S01'
              and t.c_nter is not null
              and t.ods_deleted_flg <> '1'
              and t.cf_trx_stat <> 'R'
              and trunc(to_date(cast(t.d_trx_orig as timestamp)), 'MM') = p.month_start
        )
        select
            cast('{month_start}' as date) as month_start,
            count(distinct agr_id) as sql_agr_cnt,
            count(distinct inn) as sql_inn_cnt
        from feb_clients
    """)
sql_cnt

## Сравнение с Excel (вручную)
- Сопоставьте `sql_inn_cnt` с `excel_unique_inn`.
- Сопоставьте `sql_agr_cnt` с количеством договорных строк в отчете (если в отчете grain договорный).